Use InfoQGAN structure for the data augmentations
Examples: Credit card froud dataset, Wile quality dataset, Iris dataset

In [ ]:
# Standard Libraries
import math
import pickle
import random
import numpy as np

# Data Manipulation and Visualization
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')  # For saving figures
import seaborn as sns
from sklearn.manifold import TSNE
from IPython.display import clear_output
from tqdm import tqdm

# Quantum Computing
import pennylane as qml

# PyTorch Libraries and Tools
import torch
import torch.nn as nn
from torch.autograd import Variable
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader, TensorDataset

# Utility Functions
from functools import reduce
import ndtest # 2D 분포 검정에 사용
from datetime import datetime
import os
import time
from modules.utils import convert_ipynb_to_html # 현재 html파일 저장을 위해 사용
import argparse
import json
from scipy.stats import ks_2samp

train_type = "InfoQGAN"
use_mine = True if train_type == "InfoQGAN" else False

n_qubits = 6
code_qubits = 2
output_qubits = n_qubits
noise_qubits = n_qubits - code_qubits

n_features = n_qubits

n_layers = 20
BATCH_SIZE = 16
SEED = 1
epoch_num = 300

G_lr = 0.001
D_lr = 0.0001
M_lr = 0.001
coeff = 0.05

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yasserh/wine-quality-dataset")

print("Path to dataset files:", path)

# 다운받은 파일들이 있는 경로 확인
print("다운받은 파일 목록:", os.listdir(path))

# 예시로, 'winequality.csv'라는 파일이 있다고 가정합니다.
csv_file = os.path.join(path, "WineQT.csv")
train_data_df = pd.read_csv(csv_file)

# DataFrame의 처음 5행 출력
print(train_data_df.shape, train_data_df.columns)

c:\Users\minkyu\anaconda3\envs\quantum\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\minkyu\.cache\kagglehub\datasets\yasserh\wine-quality-dataset\versions\1
다운받은 파일 목록: ['data', 'WineQT.csv']
(1143, 13) Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality', 'Id'],
      dtype='object')


In [3]:
# 수치형 변수 선택
numeric_cols = train_data_df.select_dtypes(include=['float64', 'int64']).columns.tolist()

plt.figure(figsize=(15, 10))
for i, col in enumerate(numeric_cols):
    plt.subplot(math.ceil(len(numeric_cols) / 3), 3, i + 1)
    sns.histplot(train_data_df[col], kde=True, bins=15)
    plt.title(col)
plt.tight_layout()
plt.show()

C:\Users\minkyu\AppData\Local\Temp\ipykernel_26932\4156270177.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
plt.figure(figsize=(12, 8))
corr = train_data_df.corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()


C:\Users\minkyu\AppData\Local\Temp\ipykernel_26932\140779888.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
numeric_cols = train_data_df.select_dtypes(include=['float64', 'int64']).columns.tolist()
data_numeric = train_data_df[numeric_cols].values

# t-SNE 적용 (2차원으로 변환)
tsne = TSNE(n_components=2, random_state=42)
tsne_result = tsne.fit_transform(data_numeric)

# t-SNE 결과를 DataFrame으로 저장
tsne_df = pd.DataFrame(tsne_result, columns=['Component 1', 'Component 2'])
tsne_df['quality'] = train_data_df['quality']

# t-SNE 결과를 시각화 (quality에 따른 색상 표시)
plt.figure(figsize=(10, 7))
sns.scatterplot(x='Component 1', y='Component 2', hue='quality', palette='viridis', data=tsne_df, alpha=0.7)
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.title("t-SNE Visualization of Numeric Data (Colored by Quality)")
plt.legend(title='Quality')
plt.show()


C:\Users\minkyu\AppData\Local\Temp\ipykernel_26932\1742477032.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
sns.pairplot(train_data_df)
plt.show()

C:\Users\minkyu\AppData\Local\Temp\ipykernel_26932\2537029354.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
from sklearn.feature_selection import SelectKBest, f_regression

# quality를 target으로, 나머지 속성들을 feature로 지정
X = train_data_df.drop('quality', axis=1)
y = train_data_df['quality']

# features개수 제한
selector = SelectKBest(score_func=f_regression, k=n_features-1) # quality 빼고 나머지 n-1개
selector.fit(X, y)

# 선택된 feature의 boolean mask와 이름을 추출
mask = selector.get_support()
selected_features = X.columns[mask].tolist()

print("선택된 속성들:", selected_features)

# 선택된 속성들과 quality를 포함하는 데이터셋 구성
train_data_selected = train_data_df[selected_features + ['quality']]

# 학습 데이터 변수 생성
train_in = train_data_selected.to_numpy()

선택된 속성들: ['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates', 'alcohol']


In [8]:
# 수치형 변수 선택
numeric_cols = train_data_selected.select_dtypes(include=['float64', 'int64']).columns.tolist()

plt.figure(figsize=(15, 10))
for i, col in enumerate(numeric_cols):
    plt.subplot(math.ceil(len(numeric_cols) / 3), 3, i + 1)
    sns.histplot(train_data_selected[col], kde=True, bins=15)
    plt.title(col)
plt.tight_layout()
plt.show()

C:\Users\minkyu\AppData\Local\Temp\ipykernel_26932\2277923711.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
plt.figure(figsize=(12, 8))
corr = train_data_selected.corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()


C:\Users\minkyu\AppData\Local\Temp\ipykernel_26932\2529304331.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


골라낸 n+1 개의 column 만으로 고전 모델 학습 진행

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

# feature와 target 분리
X = train_data_selected.drop('quality', axis=1)
y = train_data_selected['quality']

# 데이터를 학습셋과 테스트셋으로 분리 (예: 80% 학습, 20% 테스트)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# RandomForestClassifier를 이용해 모델 학습
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# 테스트셋에 대해 예측
y_pred = model.predict(X_test)

# 정확도 계산
accuracy = accuracy_score(y_test, y_pred)
print("모델 정확도:", accuracy)

# 예측 결과와 실제값으로 Confusion Matrix 계산
cm = confusion_matrix(y_test, y_pred)

# Confusion Matrix 시각화
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("pred quality")
plt.ylabel("real quality")
plt.title("Wine Quality prediction Confusion Matrix")
plt.show()

모델 정확도: 0.6550218340611353


C:\Users\minkyu\AppData\Local\Temp\ipykernel_26932\3474430690.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


모델들 불러오기

In [ ]:
# setting torch device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
train_tensor = torch.tensor(train_in, dtype=torch.float32).to(device)

dev = qml.device("default.qubit", wires=n_qubits)

import importlib
from modules import QGAN, Discriminator, MINE  # 초기 import
importlib.reload(QGAN)  # 모듈 갱신
importlib.reload(Discriminator)  # 모듈 갱신
importlib.reload(MINE)  # 모듈 갱신

# 생성자 파라미터 초기화 및 모듈 불러오기
generator_initial_params = Variable(torch.tensor(np.random.normal(-np.pi/2 , np.pi/2, (n_layers, n_qubits, 1))), requires_grad=True)
generator = QGAN.QGAN2(n_qubits, output_qubits, n_layers, generator_initial_params, dev)

# 판별자, MINE 초기화
discriminator = Discriminator.LinearDiscriminator(input_dim = output_qubits, hidden_size=128)
mine = MINE.LinearMine(code_dim=code_qubits, output_dim=output_qubits)

G_opt = torch.optim.Adam([generator.params], lr=G_lr)
D_opt = torch.optim.Adam(discriminator.parameters(), lr=D_lr)
M_opt = torch.optim.Adam(mine.parameters(), lr=M_lr)

cpu


In [ ]:
def bitwise_sums(arr):
    n = len(arr).bit_length() - 1  # 비트 길이를 계산하여 반복 횟수를 정함
    sums = torch.zeros(n, dtype=arr.dtype, device=arr.device)  # 결과를 저장할 텐서
    for bit in range(n):
        # 조건에 맞는 인덱스 선택을 위해 i-th 비트를 검사
        mask = (torch.arange(len(arr), device=arr.device) >> bit) & 1
        sums[bit] = arr[mask.bool()].sum()  # 조건에 맞는 원소들의 합산
    return sums

def output_postprocessing(arr):
    # arr: (BATCH_SIZE, output_qubits**2)
    # return: (BATCH_SIZE, output_qubits)
    ret = torch.stack([bitwise_sums(arr[i]) for i in range(len(arr))])
    return ret

def output_rescale(ret):
    # 각 속성별로 평균과 표준편차를 학습 데이터셋과 동일하게 만듦
    train_means = train_data_selected.mean()
    train_stds = train_data_selected.std()

    train_means_tensor = torch.tensor(train_means.values, dtype=ret.dtype, device=ret.device)
    train_stds_tensor = torch.tensor(train_stds.values, dtype=ret.dtype, device=ret.device)

    ret_means = ret.mean(dim=0)
    ret_stds = ret.std(dim=0)

    ret_adjusted = ret * (train_means_tensor / ret_means)
    ret_adjusted_stds = ret_adjusted.std(dim=0)
    ret_normalized = train_means_tensor + (ret_adjusted - train_means_tensor) * (train_stds_tensor / ret_adjusted_stds)

    return ret_normalized

def output_normalization(ret):
    # 각 속성별로 z-score normalization으로 되돌림
    ret_normalized = (ret - ret.mean(dim=0)) / ret.std(dim=0)
    return ret_normalized

def generator_train_step(generator_input, use_mine = False):
    code_input = generator_input[:, -code_qubits:] # 입력중에서 code만 뽑는다. (BATCH_SIZE, code_qubits)

    generator_output = generator.forward(generator_input) # 출력을 뽑아낸다 (BATCH_SIZE, 2**output_qubits)
    generator_output = output_postprocessing(generator_output) # (BATCH_SIZE, output_qubits)
    generator_output = generator_output.to(torch.float32) # (BATCH_SIZE, output_qubits)
    
    disc_output = discriminator(output_normalization(generator_output)) # 평균과 표준편차 맞춘 뒤 판별자에 넣음
    gan_loss = torch.log(1-disc_output).mean()
    
    if use_mine:
        pred_xy = mine(code_input, generator_output)
        code_input_shuffle = code_input[torch.randperm(BATCH_SIZE)]
        pred_x_y = mine(code_input_shuffle, generator_output)
        mi = torch.mean(pred_xy) - torch.log(torch.mean(torch.exp(pred_x_y)))
        gan_loss -= coeff * mi

    return generator_output, gan_loss# TODO: 이건 분석용으로 넣어놓음.지워야 함.

disc_loss_fn = nn.BCELoss()
def disc_cost_fn(real_input, fake_input, smoothing=False):
    batch_num = real_input.shape[0]

    disc_real = discriminator(real_input)
    disc_fake = discriminator(fake_input)

    real_label = torch.ones((batch_num, 1)).to(device)
    fake_label = torch.zeros((batch_num, 1)).to(device)
    
    if smoothing:
        real_label = real_label - 0.2*torch.rand(real_label.shape).to(device)
    
    loss = 0.5 * (disc_loss_fn(disc_real, real_label) + disc_loss_fn(disc_fake, fake_label))
    
    return loss

In [13]:
def combined_tsne(A, asource, B, bsource):
    A = A.copy()
    B = B.copy()
    A["source"] = asource
    B["source"] = bsource
    combined_df = pd.concat([A, B], axis=0, ignore_index=True)
    tsne = TSNE(n_components=2, random_state=42)
    tsne_result = tsne.fit_transform(combined_df.drop("source", axis=1).values)
    combined_df["TSNE_1"] = tsne_result[:, 0]
    combined_df["TSNE_2"] = tsne_result[:, 1]
    fig = plt.figure(figsize=(10, 7))
    sns.scatterplot(x="TSNE_1", y="TSNE_2", hue="source", data=combined_df, palette=["blue", "orange"], alpha=0.5)
    plt.title(f"t-SNE Visualization: {asource} vs {bsource}")
    return fig

def visualize_output_augment(log_gen_outputs, log_gen_codes, epoch, writer, image_file_path):
    # 1. 첫 번째 플롯: 출력의 hisplot
    fig1 = plt.figure(figsize=(15, 10))
    for i, col in enumerate(train_data_selected.columns):
        plt.subplot(math.ceil(n_features / 3), 3, i + 1)
        sns.histplot(log_gen_outputs[:, i], kde=True, bins=15)
        plt.title(f"Column {col}")
    plt.tight_layout()

    # 2. 두 번째 플롯: 출력의 상관관계 heatmap
    fig2 = plt.figure(figsize=(12, 8))
    corr = np.corrcoef(log_gen_outputs, rowvar=False)
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Correlation Heatmap")

    # 세 번째 플롯: t-SNE 시각화
    fig3 = combined_tsne(train_data_selected, "train data", pd.DataFrame(log_gen_outputs, columns=train_data_selected.columns), "generated")

    # TensorBoard에 기록
    writer.add_figure(f'hisplot', fig1, epoch)
    writer.add_figure(f'heatmap', fig2, epoch)
    writer.add_figure(f'tsnes', fig3, epoch)

    # fig1, fig2, fig3 를 image file로 저장
    fig1.savefig(f'{image_file_path}/hist_epoch_{epoch}.png')
    fig2.savefig(f'{image_file_path}/corr_epoch_{epoch}.png')
    fig3.savefig(f'{image_file_path}/tsne_epoch_{epoch}.png')

    # 메모리 관리를 위해 plt를 닫음
    plt.close(fig1)
    plt.close(fig2)
    plt.close(fig3)

In [ ]:
current_time = datetime.now().strftime("%b%d_%H_%M_%S")  # "Aug13_14_12" 형식
trial_name = f"WineQT_{train_type}_nq{n_qubits}_nl{n_layers}_{current_time}"
save_dir = f"./runs/{trial_name}"
scalar_save_path = os.path.join(save_dir, f"{trial_name}.csv")
image_save_dir = os.path.join(save_dir, "images")
numpy_save_dir = os.path.join(save_dir, "numpy")
param_save_dir = os.path.join(save_dir, "params")
os.makedirs(image_save_dir, exist_ok=True)
os.makedirs(numpy_save_dir, exist_ok=True)
os.makedirs(param_save_dir, exist_ok=True)

convert_ipynb_to_html('augment_train.ipynb', os.path.join(save_dir, "augment_train.html"))

# CSV 파일 초기화 (헤더 작성)
if not os.path.exists(scalar_save_path):
    df = pd.DataFrame(columns=['epoch', 'D_loss', 'G_loss', 'MI', 'time'] + 
                  [f'Corr/code{i}-{axis}' for i in range(code_qubits) for axis in train_data_selected.columns] +
                  [f'{feature_name}_D_ks' for feature_name in train_data_selected.columns] +
                  [f'{feature_name}_p_value' for feature_name in train_data_selected.columns])

# TensorBoard SummaryWriter 초기화
writer = SummaryWriter(log_dir=save_dir)

epoch_num = 300
start_time = time.time()

train_tensor_normalized = (train_tensor - train_tensor.mean(dim=0)) / train_tensor.std(dim=0) # 각 속성별로 z-score normalization
train_loader = DataLoader(
    TensorDataset(train_tensor_normalized),
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True,
    drop_last=True  # 마지막 배치 크기가 작으면 무시
)

for epoch in range(1, epoch_num+1):
    G_loss_sum = 0.0
    D_loss_sum = 0.0
    mi_sum = 0.0
    batch_num = len(train_in) // BATCH_SIZE
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epoch_num}", unit="batch")

    # 그림 그릴때 필요하다
    gen_outputs = [] # (데이터수, n_features) QGAN으로 생성한 데이터
    gen_codes = [] # (데이터수, code_qubits) 점 찍는데 들어간 code들

    for batch_idx, (batch,) in enumerate(pbar):  # batch unpacking
        # train generator
        generator_seed = torch.rand((BATCH_SIZE, n_qubits))*2-1
        generator_output, generator_loss = generator_train_step(generator_seed, use_mine=use_mine)
        G_opt.zero_grad()
        generator_loss.requires_grad_(True)
        generator_loss.backward()
        G_opt.step()
        
        generator_output = generator_output.detach().to(torch.float32)
        # train discriminator
        fake_input = output_normalization(generator_output)
        disc_loss = disc_cost_fn(batch, fake_input, smoothing=False)
        D_opt.zero_grad()
        disc_loss.requires_grad_(True)
        disc_loss.backward()
        D_opt.step()

        # train mine
        code_input = generator_seed[:, -code_qubits:] # (BATCH_SIZE, code_qubits) 코드만 추출
        pred_xy = mine(code_input, generator_output)
        code_input_shuffle = code_input[torch.randperm(BATCH_SIZE)]
        pred_x_y = mine(code_input_shuffle, generator_output)
        mi = -torch.mean(pred_xy) + torch.log(torch.mean(torch.exp(pred_x_y)))
        M_opt.zero_grad()
        mi.requires_grad_(True)
        mi.backward()
        M_opt.step()

        D_loss_sum += disc_loss.item()
        G_loss_sum += generator_loss.item()
        mi_sum -= mi.item() # (-1)곱해져 있어서 빼야함.

        gen_outputs.append(output_rescale(fake_input).numpy()) # 결과 기록할 땐 실제 데이터 범위로 만듦
        gen_codes.append(code_input.numpy())

        pbar.set_postfix({'G_loss': G_loss_sum/(batch_idx+1), 'D_loss': D_loss_sum/(batch_idx+1), 'MI': mi_sum/(batch_idx+1)})

    # G_scheduler.step()
    # D_scheduler.step()
    # M_scheduler.step()
    
    gen_outputs = np.concatenate(gen_outputs, axis=0) # (train_num, n_features)
    gen_outputs[:, -1] = np.round(gen_outputs[:, -1]) # quality는 정수로 반올림

    gen_codes = np.concatenate(gen_codes, axis=0) # (train_num, 2)

    D_loss, G_loss, mi = D_loss_sum/batch_num, G_loss_sum/batch_num, mi_sum/batch_num

    writer.add_scalar('Loss/d_loss', D_loss, epoch)
    writer.add_scalar('Loss/g_loss', G_loss, epoch)
    writer.add_scalar('Loss/mi', mi, epoch)
    D_ks_dict = {}
    p_value_dict = {}
    for i in range(n_features):
        stat, p_value = ks_2samp(gen_outputs[:, i], train_in[:, i])
        feature_name = train_data_selected.columns[i]
        writer.add_scalar(f'D_ks/{feature_name}', stat, epoch)
        writer.add_scalar(f'p_value/{feature_name}', p_value, epoch)
        D_ks_dict[f'{feature_name}_D_ks'] = stat
        p_value_dict[f'{feature_name}_p_value'] = p_value
    

    correlation_matrix = np.zeros((code_qubits, n_features))      # 각 code와 basis state간의 상관관계
    for i in range(code_qubits):
        for j in range(n_features):
            correlation_matrix[i, j] = np.corrcoef(gen_codes[:, i], gen_outputs[:, j])[0, 1]
            writer.add_scalar(f'Corr/code{i}-{train_data_selected.columns[j]}', correlation_matrix[i, j], epoch)

    # calculation all angle between correlation_matrix[i, :] and correlation_matrix[j, :]
    angle_sum = 0
    for i in range(code_qubits):
        for j in range(i+1, code_qubits):
            cos_theta = np.dot(correlation_matrix[i, :], correlation_matrix[j, :]) / (np.linalg.norm(correlation_matrix[i, :]) * np.linalg.norm(correlation_matrix[j, :]))
            theta_degrees = np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0)))
            theta_degrees = min(theta_degrees, 180 - theta_degrees) # 예각으로 변환
            writer.add_scalar(f'Angle/angle{i}-{j}', theta_degrees, epoch)
            angle_sum += theta_degrees

    if code_qubits > 1:
        angle_avg = angle_sum / (code_qubits * (code_qubits - 1) / 2)
        writer.add_scalar('Angle/Code_Angle', angle_avg, epoch)

    # 스칼라 값 CSV로 덮어쓰기 저장
    file_exists = os.path.isfile(scalar_save_path)
    new_data = pd.DataFrame({
        'epoch': [epoch],
        'D_loss': [D_loss],
        'G_loss': [G_loss],
        'MI': [mi],
        **D_ks_dict,
        **p_value_dict,
        'time': [int((time.time() - start_time)*1000)],
        **{f'Corr/code{i}-{axis}': [correlation_matrix[i, j]] for i in range(code_qubits) for j, axis in enumerate(train_data_selected.columns)},
    })

    new_data.to_csv(scalar_save_path, mode='a',  header=not file_exists)
    
    visualize_output_augment(gen_outputs, gen_codes, epoch, writer, image_save_dir) # save fig here

    # 각 epoch마다 numpy의 savetxt를 사용하여 저장
    output_file_path = os.path.join(numpy_save_dir, f"gen_outputs_epoch_{epoch}.txt")
    codes_file_path = os.path.join(numpy_save_dir, f"gen_codes_epoch_{epoch}.txt")

    np.savetxt(output_file_path, gen_outputs)
    np.savetxt(codes_file_path, gen_codes)
    torch.save(generator.params, f'{param_save_dir}/generator_params_epoch{epoch}.pth') # QGAN 파라미터 저장

HTML 파일이 ./runs/WineQT_InfoQGAN_nq6_nl20_Feb19_13_10_05\augment_train.html에 저장되었습니다.


Epoch 300/300: 100%|██████████| 71/71 [02:08<00:00,  1.82s/batch, G_loss=-0.744, D_loss=0.671, MI=1.26]


In [15]:
{f'Corr/code{i}-{axis}': [correlation_matrix[i, j]] for i in range(code_qubits) for j, axis in enumerate(train_data_selected.columns)}

{'Corr/code0-volatile acidity': [np.float64(-0.7326195581953734)],
 'Corr/code0-citric acid': [np.float64(0.5695056744710753)],
 'Corr/code0-total sulfur dioxide': [np.float64(0.17852831061840793)],
 'Corr/code0-sulphates': [np.float64(-0.23724407879578874)],
 'Corr/code0-alcohol': [np.float64(0.39306655909507127)],
 'Corr/code0-quality': [np.float64(0.2572299920027591)],
 'Corr/code1-volatile acidity': [np.float64(-0.3265800401022327)],
 'Corr/code1-citric acid': [np.float64(0.47768663449653886)],
 'Corr/code1-total sulfur dioxide': [np.float64(-0.34201878264294316)],
 'Corr/code1-sulphates': [np.float64(0.3651004259754557)],
 'Corr/code1-alcohol': [np.float64(-0.1512031645982765)],
 'Corr/code1-quality': [np.float64(0.3204946631301078)]}